### Feature Engineering: Hybrid & Ratio Features

Objective: Create new features that capture customer behavior ratios, which are often stronger predictors of churn than raw numbers.

Data Integrity Note: 
Some required columns (e.g., attempt_Mean, totmou) were dropped during the EDA phase due to high multicollinearity. To calculate these ratios, we fetch the missing columns from the raw datasets and immediately impute any missing values (using the median) to ensure our dataset remains 100% clean.

New Features:
1. revenue_per_mou: Revenue per minute of use.
2. drop_rate: Ratio of dropped calls to total attempts.
3. care_calls_per_month: Customer care calls normalized by tenure.
4. eqpdays_per_month: Equipment age normalized by tenure.

In [20]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder

# 1. Load the cleaned data (Make sure you saved this file from eda.ipynb, or use the 'df' variable if you are in the same session)
df = pd.read_csv('../data/processed/cleaned_data_eda.csv') 

# 2. Drop ID column (useless for the model)
df_fe = df.drop(columns=['Customer_ID'], errors='ignore')

In [21]:
import pandas as pd

# 1. List of all columns needed for hybrid features
needed_cols = [
    'totrev', 'totmou', 'eqpdays', # from Client
    'drop_vce_Mean', 'attempt_Mean', 'custcare_Mean', 'months' # from Record
]

# 2. Check which columns are missing in the current df
missing_cols = [col for col in needed_cols if col not in df.columns]

if missing_cols:
    print(f"Found missing columns: {missing_cols}. Fetching them from raw data...")
    
    # 3. Fetch from Client
    client_needed = ['Customer_ID'] + [col for col in missing_cols if col in ['totrev', 'totmou', 'eqpdays']]
    if len(client_needed) > 1:
        raw_client = pd.read_csv('../data/raw/Client.csv', usecols=client_needed)
        df = df.merge(raw_client, on='Customer_ID', how='left')
        
    # 4. Fetch from Record
    record_needed = ['Customer_ID'] + [col for col in missing_cols if col in ['drop_vce_Mean', 'attempt_Mean', 'custcare_Mean', 'months']]
    if len(record_needed) > 1:
        raw_record = pd.read_csv('../data/raw/Record.csv', usecols=record_needed)
        df = df.merge(raw_record, on='Customer_ID', how='left')

    # ====================================================================
    # 5. CRITICAL STEP: Clean the newly merged columns immediately!
    # Fill any missing values with the median, just like we did in EDA.
    # ====================================================================
    for col in missing_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
            
    print("New columns fetched AND cleaned successfully (NaNs filled with median).")

# 6. Now create the new dataframe and calculate features safely
df_fe = df.copy()

# Adding +1 to denominator to avoid division by zero
df_fe['revenue_per_mou'] = df_fe['totrev'] / (df_fe['totmou'] + 1)
df_fe['drop_rate'] = df_fe['drop_vce_Mean'] / (df_fe['attempt_Mean'] + 1)
df_fe['care_calls_per_month'] = df_fe['custcare_Mean'] / (df_fe['months'] + 1)
df_fe['eqpdays_per_month'] = df_fe['eqpdays'] / (df_fe['months'] + 1)

print("Hybrid features added successfully!")

# لا تنسى تشغيل خلية الـ LabelEncoder بعد هذا الكود، ثم حفظ الملف!

Found missing columns: ['totmou', 'attempt_Mean']. Fetching them from raw data...
New columns fetched AND cleaned successfully (NaNs filled with median).
Hybrid features added successfully!


In [22]:
import numpy as np

# ==========================================
# 1. Clean Garbage & Handle Negative Values
# ==========================================
# Replace negative values in usage and revenue with 0
cols_to_check = ['totrev', 'totmou', 'mou_Mean', 'rev_Mean', 'avg3mou', 'avg6mou']
for col in cols_to_check:
    if col in df_fe.columns:
        df_fe[col] = df_fe[col].clip(lower=0)

# ==========================================
# 2. Advanced Feature Engineering (Trends)
# ==========================================
# Trend Features (The Churn Signal)
if 'avg3mou' in df_fe.columns and 'avg6mou' in df_fe.columns:
    df_fe['usage_trend_3v6'] = df_fe['avg3mou'] - df_fe['avg6mou']

if 'avg3rev' in df_fe.columns and 'avg6rev' in df_fe.columns:
    df_fe['revenue_trend_3v6'] = df_fe['avg3rev'] - df_fe['avg6rev']

# Average per month
df_fe['rev_per_month'] = df_fe['totrev'] / (df_fe['months'] + 1)
df_fe['mou_per_month'] = df_fe['totmou'] / (df_fe['months'] + 1)

# ==========================================
# 3. Smart Encoding for High Cardinality
# ==========================================
# Replace Label Encoding for 'crclscod' with Frequency Encoding
if 'crclscod' in df_fe.columns:
    freq = df_fe['crclscod'].value_counts()
    df_fe['crclscod'] = df_fe['crclscod'].map(freq)

# ==========================================
# 4. Handle Outliers (Capping top 1%)
# ==========================================
# This prevents extreme values from breaking the trees
for col in ['totrev', 'totmou', 'eqpdays']:
    if col in df_fe.columns:
        upper_limit = df_fe[col].quantile(0.99)
        df_fe[col] = np.where(df_fe[col] > upper_limit, upper_limit, df_fe[col])

print("Advanced Data Cleaning, Trends, and Outlier Capping Completed!")

Advanced Data Cleaning, Trends, and Outlier Capping Completed!


In [23]:
# Encode categorical columns (Convert text to numbers)
le = LabelEncoder()
cat_cols = df_fe.select_dtypes(include=['object', 'string', 'category']).columns

for col in cat_cols:
    df_fe[col] = df_fe[col].astype(str)
    df_fe[col] = le.fit_transform(df_fe[col])
# Save the final data for modeling
os.makedirs('../data/processed', exist_ok=True)
df_fe.to_csv('../data/processed/data_for_modeling.csv', index=False)

print(f"Data is ready! Shape: {df_fe.shape}")
print(f"Total missing values: {df_fe.isnull().sum().sum()}")
print("File saved to: ../data/processed/data_for_modeling.csv")

Data is ready! Shape: (100000, 102)
Total missing values: 0
File saved to: ../data/processed/data_for_modeling.csv
